In [0]:
trip_df = spark.read.parquet(
    "/Volumes/workspace/default/raw_data/yellow_tripdata_2024-01.parquet"
)

In [0]:
from pyspark.sql.functions import current_timestamp, lit

bronze_trip_df = (
    trip_df
    .withColumn("source_file",
                lit("yellow_tripdata_2024-01.parquet"))
    .withColumn("ingestion_timestamp",
                current_timestamp())
)

In [0]:
display(
    bronze_trip_df.select(
        "VendorID",
        "trip_distance",
        "source_file",
        "ingestion_timestamp"
    )
)

In [0]:
bronze_trip_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.bronze_trips")

In [0]:
spark.sql("""
SELECT COUNT(*)
FROM workspace.default.bronze_trips
""").show()

In [0]:
%sql
SELECT *
FROM workspace.default.bronze_trips
LIMIT 10

In [0]:
%sql
SELECT
    COUNT(*) AS total_trips,
    AVG(fare_amount) AS avg_fare,
    MAX(fare_amount) AS highest_fare,
    MIN(fare_amount) AS lowest_fare
FROM workspace.default.bronze_trips

In [0]:
%sql
SELECT
    COUNT(*) AS bad_fares
FROM workspace.default.bronze_trips
WHERE fare_amount < 0

In [0]:
%sql
SELECT
    fare_amount,
    total_amount,
    trip_distance,
    passenger_count
FROM workspace.default.bronze_trips
WHERE fare_amount < 0
LIMIT 20

In [0]:
%sql
SELECT
    COUNT(*) AS total_records,
    COUNT(CASE WHEN fare_amount < 0 THEN 1 END) AS negative_fares
FROM workspace.default.bronze_trips